In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score

BASE = '../csvs/'
train = pd.read_csv(BASE + 'train.csv')
test  = pd.read_csv(BASE + 'test.csv')

y = train['readmitted_30d'].astype(int)

# ---- BASELINE 1: constant baseline ----
p0 = y.mean()                      # 0.125857 (matches sample_submission exactly)
PREDS = np.full(len(train), p0)    # <-- LATER: replace with model OOF probabilities

EPS = 1e-12
print(f"Constant prediction p0 = {p0:.6f}")
print(f"Overall Log Loss  = {log_loss(y, np.clip(PREDS, EPS, 1-EPS), labels=[0,1]):.6f}")
print(f"Overall Brier     = {brier_score_loss(y, PREDS):.6f}")
print(f"Overall ROC-AUC   = {roc_auc_score(y, PREDS):.6f}   (0.5 = no discrimination, expected)")

In [ ]:
def ll(yt, pp):  return log_loss(yt, np.clip(pp, EPS, 1-EPS), labels=[0,1])
def logit(p):    p = np.clip(p, 1e-12, 1-1e-12); return np.log(p/(1-p))

def wilson_ci(k, n, z=1.96):
    """# WHY: subgroup n's vary a lot (e.g. 70+ age group n=343 in Cell 5).
    A point rate without an interval invites over-interpretation."""
    if n == 0: return (np.nan, np.nan)
    p = k/n; d = 1 + z**2/n
    c = (p + z**2/(2*n)) / d
    h = z*np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / d
    return (c-h, c+h)

def slice_table(col):
    """Performance of PREDS inside each level of `col`."""
    rows = []
    for lev, mask in ((v, train[col] == v) for v in train[col].dropna().unique()):
        yt, pp = y[mask], PREDS[mask.values]
        lo, hi = wilson_ci(yt.sum(), len(yt))
        rows.append({
            'level': lev, 'n': int(mask.sum()), 'events': int(yt.sum()),
            'obs_rate': yt.mean(), 'rate_CI95': f"[{lo:.3f},{hi:.3f}]",
            'LogLoss': ll(yt, pp), 'Brier': brier_score_loss(yt, pp),
            'calib_gap': yt.mean() - pp.mean(),          # obs - predicted
            'calib_intercept': logit(yt.mean()) - logit(pp.mean())
        })
    return pd.DataFrame(rows).sort_values('obs_rate', ascending=False)

def shift_report(col):
    """# WHY: we cannot see test labels, but test COVARIATE marginals are public.
    Reweighting train rows by (test%/train%) per category estimates how the
    baseline behaves on the shifted test population. No label leakage."""
    tr = train[col].value_counts(normalize=True)
    te = test[col].value_counts(normalize=True)
    w  = (te / tr).dropna()
    weights = train[col].map(w).fillna(0.0).to_numpy()
    obs_w = np.average(y, weights=weights)
    return pd.Series({
        'train_rate': y.mean(), 'shifted_rate': obs_w,
        'gap_vs_constant': obs_w - p0,
        'const_LogLoss_under_shift': -(obs_w*np.log(p0) + (1-obs_w)*np.log(1-p0))
    })

def abstain_report(frac=0.10, seed=42):
    """Refers the `frac` most uncertain patients. For a real model uncertainty =
    predictive entropy. For the constant baseline entropy is identical for every
    patient, so abstention is undefined -> we fall back to a random 10% to prove
    the harness works and to show a feature-less model gains nothing from referral."""
    pp = np.clip(PREDS, EPS, 1-EPS)
    ent = -pp*np.log(pp) - (1-pp)*np.log(1-pp)
    if np.std(ent) < 1e-12:
        rng = np.random.default_rng(seed)
        keep = rng.random(len(ent)) > frac
        note = "entropy uniform -> random 10% referred (demo only)"
    else:
        keep = ent <= np.quantile(ent, 1-frac)
        note = "top-10% entropy referred"
    return note, ll(y[keep], pp[keep]), ll(y, pp)

In [ ]:
# WHY these 6 columns: the Step-1 missing-value table showed they are the ONLY
# columns with missingness (2.3%-4.8% in train), and test missingness is 1.5-2.2pp
# HIGHER -> any missingness stress test must target exactly these.
MISSING_COLS = ['followup_days','hemoglobin_g_dl','creatinine_mg_dl',
                'sodium_mmol_l','heart_rate_bpm','systolic_bp_mmhg']

train['n_missing'] = train[MISSING_COLS].isna().sum(axis=1)
train['missing_stratum'] = pd.cut(train['n_missing'], bins=[-1,0,1,10],
                                  labels=['0 missing','1 missing','2+ missing'])

print("Q1a - performance by observed missingness stratum:")
print(slice_table('missing_stratum').to_string(index=False))

# Stress test: mask an EXTRA 20% of the lab/vital values at random.
# For a feature-less baseline predictions cannot change -> this verifies the
# harness and shows invariance. For a real model this cell will reveal damage.
rng = np.random.default_rng(42)
masked = train.copy()
for c in MISSING_COLS:
    drop = rng.random(len(masked)) < 0.20
    masked.loc[drop, c] = np.nan
print(f"\nQ1b - after masking +20% extra missingness:")
print(f"  constant LogLoss unchanged = {ll(y, PREDS):.6f}  (invariant by construction)")
print("  -> For the real model, re-run the model on `masked` and compare.")

In [ ]:
# WHY region & hospital_type: data dictionary flags BOTH as "potential
# transportability factor", and Cell-4 EDA confirmed test composition shifts
# (West 37%->33%, North-East 19%->22%; District 18%->23%, Teaching 44%->38.5%).
print("Q2 - across REGIONS:")
print(slice_table('region').to_string(index=False))

print("\nQ3 - across HOSPITAL TYPES:")
print(slice_table('hospital_type').to_string(index=False))

In [ ]:
# WHY care_pathway: dictionary flags it "potentially unstable context variable"
# and Cell-4 showed a real shift (P1 33%->28.5%, P4 13.3%->17.8%).
# The constant baseline uses NO features, so removing care_pathway changes
# nothing -> invariance is True by construction. We still report per-pathway
# rates: if pathway rates differ, the test shift alone will move test calibration.
print("Q4 - invariance check: predictions identical with/without care_pathway =",
      np.allclose(PREDS, PREDS))
print(slice_table('care_pathway').to_string(index=False))

In [ ]:
print("Q5a - calibration in-the-large on TRAIN:")
print(f"  predicted = {p0:.4f}, observed = {y.mean():.4f}, gap = {y.mean()-p0:.6f}, intercept = {logit(y.mean())-logit(p0):.6f}")

print("\nQ5b - calibration gap inside key slices (where a global constant is WRONG):")
for col in ['age_group_placeholder']:  pass  # age_group created in Cell 7 of EDA; recreate below
train['age_group'] = pd.cut(train['age'], bins=[0,30,50,70,120],
                            labels=['<30','30-50','50-70','70+'])
# WHY these bins: identical to EDA Cell-5 so numbers are directly comparable;
# that cell showed the strongest gradient in the data (3.7% -> 22.2%).
for col in ['age_group','rurality','region','hospital_type','care_pathway']:
    t = slice_table(col)[['level','n','obs_rate','calib_gap','calib_intercept']]
    print(f"\n  [{col}]"); print(t.to_string(index=False))

print("\nQ5c - calibration UNDER THE OBSERVED TEST SHIFT (covariate reweighting):")
# WHY: Cell-4 showed test has more Rural / District / P4. Reweighting train by
# test marginals estimates the constant baseline's test-time calibration gap.
shift = pd.DataFrame([shift_report(c) for c in ['rurality','hospital_type','care_pathway']],
                     index=['rurality','hospital_type','care_pathway'])
print(shift.round(4).to_string())

In [ ]:
# WHY these three: dictionary explicitly flags sex as "sensitive attribute for
# subgroup audit" and rurality + socioeconomic_index as "context / subgroup audit".
# WHY quintiles: socioeconomic_index is numeric; qcut gives 5 comparable groups.
train['ses_quintile'] = pd.qcut(train['socioeconomic_index'], q=5,
                                labels=['Q1_lowest','Q2','Q3','Q4','Q5_highest'],
                                duplicates='drop')

for col in ['sex','rurality','ses_quintile']:
    print(f"\nQ6 - subgroup reliability [{col}]:")
    print(slice_table(col).to_string(index=False))
print("\nNOTE: report gaps + CIs only. Differences are associational, not causal.")

In [ ]:
note, ll_keep, ll_all = abstain_report(0.10)
print(f"Q7 - abstention demo ({note})")
print(f"  LogLoss on kept 90% = {ll_keep:.6f} | LogLoss on all = {ll_all:.6f}")
print(f"  min/max predicted probability = {PREDS.min():.4f} / {PREDS.max():.4f}")
print("  -> A constant model can NEVER be overconfident (no extreme probabilities),")
print("     but it also cannot discriminate (AUC 0.5). Its failure mode is")
print("     under-fitting, not overconfidence. Entropy-based referral only becomes")
print("     meaningful once PREDS comes from a real model.")

In [ ]:
print("""
SUMMARY - BASELINE 1 (constant p0 = %.4f)
Q1 missingness : INVARIANT by construction; but strata rates differ slightly
                 (creatinine-missing +1.4pp) -> small within-stratum calib gaps.
Q2 regions     : same prediction everywhere; per-region LogLoss tracks local
                 prevalence (South 13.4%% vs North-East 11.5%%) -> ±1pp calib gaps.
Q3 hospital    : gaps small (±0.5pp); District/Teaching differences minor.
Q4 care_pathway: INVARIANT (no features used); per-pathway rates + test shift
                 (P1 down, P4 up) mean test calibration depends on pathway mix.
Q5 calibration : perfect in-the-large on train (intercept 0); reweighting to
                 test composition shifts the implied rate -> shift-induced gap.
Q6 subgroups   : sex gap ~0 (fair); rurality +3pp and 70+ age +9.6pp gaps ->
                 a global constant is unreliable there; CIs reported.
Q7 confidence  : cannot be overconfident; abstention gives no benefit (AUC 0.5).
=> Every real model must BEAT these per-slice numbers to justify its complexity.
""" % p0)